# Handling Missing Data for Advanced ML

Data Science, ML?? data cleansing? ?? ??

## 3 major types of missing values
1. Missing Completely At Random (MCAR)
2. Missing At Random (MAR)
3. Missing Not At Random (MNAR)

### 1. Missing Completely At Random (MCAR)
: Missing value? ?? ???? ??? ??? ?? ??

e.g.
- ???? ????? ????? ??? ?? ?? ??? ??? ? or ????

$\Rightarrow$ ??? ???? ??? ??? ?? **??? ??? ??**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
plt.rc('font', family='NanumGothic')
%matplotlib inline
# create a synthetic dataset

# measured predictors x1, x2
x1 = np.sort(np.random.uniform(-2.0, 3.0, 100))  # random real number
x2 = np.random.choice([0,1], size=(100,))  # random binary numbers

x = (x1, x2)
# measured response y
y = 2 * x1 + 1
y = y + np.random.normal(0, 1, 100)

# sample
pd.DataFrame({'x1': x1, "x2": x2, "y": y}).head()

In [ ]:
# To illustrate MCAR, remove half of the values for x1

## randomly mark half of x1 smaples as missing MCAR
# independed of any information recorded
idx_mcar = np.random.choice([0,1], size=(100,)) == 1  # ndarray (100,), values are boolean type

plt.scatter(x1[~idx_mcar], y[~idx_mcar], label = 'data')  # ~idx_mcar == False
plt.scatter(x1[idx_mcar], y[idx_mcar], label='missing (MCAR)', color="blue", alpha=0.2)  # idx_mcar == True
plt.xlabel('x1')
plt.ylabel('y')
plt.legend()
plt.title("Missing Completely at Random (MCAR)")

red mark(MCAR)? $x_1$? ??? ?? ?? ??? ?? ??? ??. 

$y = 2x_1 + 1 + np.random.noraml(0,1,100)$

## Give Example from Titinic Data

In [ ]:
train_df = pd.read_csv("data/train.csv")
test_df = pd.read_csv("data/test.csv")

total = train_df.isnull().sum().sort_values(ascending=False)
percent = (train_df.isnull().sum() / train_df.shape[0]).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=["Total", "Percent"])
percent_data = percent.head(20)
percent_data.plot(kind="bar", figsize=(8,6), fontsize=10)
plt.xlabel("", fontsize=20)
plt.ylabel("", fontsize=20)
plt.title("Total Missing Value (%)", fontsize=20)

In [ ]:
import missingno as msno

missingdata_df = train_df.columns[train_df.isnull().any()].tolist()
msno.heatmap(train_df[missingdata_df], figsize=(8,6))
plt.title("Correlation with Missing Values", fontsize=20)

#### ????? ??? ?? ??? ??. ?? ?? ?? ???? ?? ? MCAR? ????.

### 2. Missing At Random (MAR)
: ??? ??? ?? ??? ?? ???? ???? ??

e.g.
 - ???? ?? ??(?? ??? ??? ????)? ???? ?? ?? $\rightarrow$ ??(?, ?)? ??(yes, no / 1, 0)? ??

In [ ]:
# Missing at Random (MAR)

# randomly mark half of x1 samples as missing MAR
# depending on value of recorded predictor x2

idx_mar = x2 == 1
fig, ax = plt.subplots(1, 2, figsize=(15,5))
ax[0].scatter(x1, y, label='data')
ax[0].scatter(x1[idx_mar], y[idx_mar], label='missing', color='red')
ax[0].set_xlabel('x1')
ax[0].set_ylabel('y')
ax[0].legend()
ax[0].set_title('Missing at Random (MAR)')

ax[1].vlines(x1[x2 == 1], 0, 1, color='black')
ax[1].set_xlabel('x1')
ax[1].set_ylabel('x2')
ax[1].set_title('dependent predictor - measured')

red-marked(MAR)? $x_2$? ?????. $x_2$? 1?? $x_1$? missing value??.

### 3. Missing Not At Random (MNAR)
: ??? ?? ?? (???? ??)??? ??? ?? ?? (MCAR, MAR ?? ??)

e.g.
- ?? ???? ????? ??? ?? ??? ? ?? ??? ?? ??? ??? ???? ??. 
- ???? ?? ??? ??? ??? ??? ???? ?? ??

In [ ]:
### Missing Not at Random (MNAR)

# randomly mark half of x1 samples as missing MNAR
# depending on unrecorded predictor x3
x3 = np.random.uniform(0, 1, 100)  # x3? ???? ?? ??? ?? (??? ?? ??)
idx_mnar = x3 > .5  
print(len(x3[idx_mnar == True]))

fig, ax = plt.subplots(1,2, figsize=(15, 5))
ax[0].scatter(x1, y, label="data")
ax[0].scatter(x1[idx_mnar], y[idx_mnar], label='missing', color='red')
ax[0].set_xlabel('x1')
ax[0].set_ylabel('y')
ax[0].legend()
ax[0].set_title("Missing Not at Random (MNAR)")
ax[1].scatter(x1, x3, color="black")
ax[1].axhline(.5, -3, 3)
ax[1].set_xlabel('x1')
ax[1].set_ylabel('x3')
ax[1].set_title('dependent preditor - not measured')

## 3 Main Approaches to Compensate for Missing Values
????? ???? MCAR, MAR, MNAR?? ??? ???.
**???? MAR ??? ?????.**

#### 1. Imputation using **0, mean, median** (for quantitative variables) or **most frequent value** (for categorical variables)


#### 2. Imputation using a **randomly selected value**

: ???? ???? ?? ???? ??


#### 3. Imputation with a **model**

: ?? ??? ?? ??(?? ??, ?? ??)? ???? ? ??? ??? ??? ??? ??
    
    
- 3.1 Model-based imputation with uncertainty
    
: ?? feature? ??? ?? ??? ???? ??
    
    
- 3.2 Model-based progressive imputation
    
: ??? ????? ????? ??? ?? ???? ???? ??
    
#### 4. Additional methods: Stochastic Regression, Multiple Imputation, Datawig, Hot-Deck imputation, Extrapolation, Interpolation, Listwise Deletion

## A Practical Guide for Hadling Missing Values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pima_df = pd.read_csv("data/missing_data_pima-indians-diabetes.csv")

response = "Outcome"
predictors = pima_df.columns.difference([response]).values

pima_df.describe()

In [ ]:
pima_df.head()

- Pregnancies: ?? ??
- Glucose: ??? ?? (?? ??? ?? ?????)
- BloodPressure: ??? ??(?? ??)(mm Hg)
- SkinThickness: ?? ?? (??)(mm)
- Insulin: ?? ???(mu U/ml)
- BMI: Body mass index
- DiabetesPedigreeFunction: ??? ??
- Age: ??(years)
- Outcome (1 for early onset of diabetes within five years, 0 for not)

In [ ]:
# Convert Features to Korean
pima_df.columns = ["?? ??", "??? ??", "??", "?? ??", "???", "BMI", "??? ??", "??", "Outcome"]
pima_df

### First step: Properly flagging missing values

In [ ]:
# Check how many missing values in pima_df
pima_df.isnull().sum(), pima_df.isna().sum()

"??? ??", "??", "?? ??", "???", "BMI" ? 0? ?? ? ??. NaN, Null or None ?? 0?? ???? ??. 

NaN?? ???? ??? ??? ??? ? ??? ?? ? ??.

In [ ]:
# build preprocessing function
def clean_data(df_raw, cols_with_zeros=['??? ??', '??',
                                       '?? ??', '???', 'BMI'],
                                       response=['Outcome']):
    df = df_raw.copy()
    
    # replace zero with NaN in features
    df[cols_with_zeros] = df[cols_with_zeros].replace(0, np.nan)
    
    # remove \ and } from response
    df = df.replace(to_replace=r'\\|\}', value='', regex=True)
    
    # change response data type to int
    df[response] = df[response].astype('int')
    
    return df

pima_df_cleaned = clean_data(pima_df)
pima_df_cleaned.head()

In [ ]:
print("Proportion of missing values")
missing_values_count = pima_df_cleaned.isna().sum()*100/pima_df_cleaned.shape[0]
features_with_missing_values = missing_values_count[missing_values_count>0].index.values
missing_values_count

In [ ]:
total = pima_df_cleaned.isnull().sum().sort_values(ascending=False)
percent = (pima_df_cleaned.isnull().sum() / pima_df_cleaned.shape[0]).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=["Total", "Percent"])
percent_data = percent.head(20)
percent_data.plot(kind="bar", figsize=(8,6), fontsize=10)
plt.xlabel("", fontsize=20)
plt.ylabel("", fontsize=20)
plt.title("Total Missing Value (%)", fontsize=20)

### Second step: Statistical tests of Hypothesis (MAR, MCAR, MNAR)

In [ ]:
pima_df[(pima_df["?? ??"] == 0) & (pima_df["???"] != 0)]

"?? ??"? ???? "???"? ??

In [ ]:
pima_df_cleaned[pima_df_cleaned["???"].isna()].head()

In [ ]:
import missingno as msno

missingdata_df = pima_df_cleaned.columns[pima_df_cleaned.isnull().any()].tolist()
msno.heatmap(pima_df_cleaned[missingdata_df], figsize=(8,6))
plt.title("Correlation with Missing Values", fontsize=20)

"??" ???? "?? ??", "BMI"? ?? ??? ????.

### Third step: choose the most appropriate technique to handle missing values

#### Strategies for handling missing data

- Drop strategy
- Mean strategy
- Median strategy & most-frequent strategy
- Model-based strategy: uses the features without missing values for training kNN regression models
    - basic mode: Age, DiabetesPedigreeFunction, Outcome, Pregnancies (not NaN) ? ???? ?? features? ??
    - progressive mode: ??? feature ?? missing values ???, ? ??? feature? ???? ?? featrue? ??
    
??? data? ????? 0 ~ 1 ??? ???? ? ??. regularized logistic regression? ??? ??? performance? ?????.
 feature? ??? ?? ??? ?? penalty? ?? ??? ? ??.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import MinMaxScaler

# function for KNN model-based imputation of missing values using features without NaN as predictors
def impute_model_basic(df):
    cols_nan = df.columns[df.isna().any()].tolist()  # '??? ??', '??', '?? ??', '???', 'BMI'
    cols_no_nan = df.columns.difference(cols_nan).values  # '?? ??', '??? ??', '??', 'Outcome'
    for col in cols_nan:
        test_data = df[df[col].isna()]  # target data (Missing values)
        train_data = df.dropna()  # training data
        knr = KNeighborsRegressor(n_neighbors=5).fit(train_data[cols_no_nan], train_data[col])  # .fit(training_data, target_data)
        df.loc[df[col].isna(), col] = knr.predict(test_data[cols_no_nan])
    return df

# function for KNN model-based imputation of missing values using features without NaN as predictors, 
# including progressively added imputed features
def impute_model_progressive(df):
    cols_nan = df.columns[df.isna().any()].tolist()
    cols_no_nan = df.columns.difference(cols_nan).values
    # '??? ??', '??', '?? ??', '???', 'BMI' ??? ??? ??? ?? ??? ?? Missing Value ??
    while len(cols_nan) > 0:  
        col = cols_nan[0]
        test_data = df[df[col].isna()]
        train_data = df.dropna()
        knr = KNeighborsRegressor(n_neighbors=5).fit(train_data[cols_no_nan], train_data[col])
        df.loc[df[col].isna(), col] = knr.predict(test_data[cols_no_nan])
        cols_nan = df.columns[df.isna().any()].tolist()
        cols_no_nan = df.columns.difference(cols_nan).values
    return df

# function for imputing missing data according to a given impute_strategy:
#  drop_rows: drop all rows with one or more missing values
#  drop_cols: drop columns with one or more missing values
#  model_basic: KNN-model-based imputation with fixed predictors
#  model_progressive: KNN-model-based imputation with progressively added predictors
#  mean, median, most_frequent: imputation with mean, median or most frequent values
#
#  cols_to_standardize: if provided, the specified columns are scaled between 0 and 1, after imputation
def impute_data(df_cleaned, impute_strategy=None, cols_to_standardize=None): 
    df = df_cleaned.copy()
    if impute_strategy == 'drop_rows':
        df = df.dropna(axis=0)
    elif impute_strategy == 'drop_cols':
        df = df.dropna(axis=1)
    elif impute_strategy == 'model_basic':
        df = impute_model_basic(df)
    elif impute_strategy == 'model_progressive':
        df = impute_model_progressive(df) 
    else:
        arr = SimpleImputer(missing_values=np.nan,strategy=impute_strategy).fit(
            df.values).transform(df.values)
        df = pd.DataFrame(data=arr, index=df.index.values, columns=df.columns.values)
    if cols_to_standardize != None:
        cols_to_standardize = list(set(cols_to_standardize) & set(df.columns.values))
        df[cols_to_standardize] = df[cols_to_standardize].astype('float')
        df[cols_to_standardize] = pd.DataFrame(data=MinMaxScaler().fit(
            df[cols_to_standardize]).transform(df[cols_to_standardize]), 
                                             index=df[cols_to_standardize].index.values,
                                             columns=df[cols_to_standardize].columns.values)
    return df

## Logistic regression with missing data
??? ?? ??? ??? ? cleaned data? logistic regression model? fit??.
- dropping rows with missing values: ???? ?? ?? ??
- dropping columns with missing values: ???? ?? ?? ??
- imputing missing values with column mean: ???? ?? ???? ??
- imputing missing values with model-based prediction: ???? ?? ?? ???? ??
- imputing missing values with progressive model-based prediction: ???? progressive ?? ?? ???? ??

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import accuracy_score
from timeit import default_timer as timer

# function for handling missing values 
# and fitting logistic regression on clean data
def logistic_regression(data, impute_strategy=None,
                        cols_to_standardize=None,
                        test_size=0.25,
                        random_state=9001):
    start = timer()
    
    # store original columns
    # ['BMI', '??', '??? ??', '???', '?? ??', '??? ??', '?? ??', '??']
    original_columns = data.columns.difference(['Outcome'])
    df_imputed = impute_data(data, impute_strategy, cols_to_standardize)
    train_data, test_data = train_test_split(df_imputed, test_size=test_size,
                                             random_state=random_state)
    
    # note which predictor columns were dropped or kept
    kept_columns = df_imputed.columns.difference(['Outcome'])
    dropped_columns = original_columns.difference(df_imputed.columns)
    original_columns = original_columns.difference(['Outcome'])
    
    # prepare tensors
    X_train = train_data.drop(columns=['Outcome'])
    y_train = train_data['Outcome']
    X_test = test_data.drop(columns=['Outcome'])
    y_test = test_data['Outcome']
    
    # model training
    logistic_model = LogisticRegressionCV(cv=10, penalty='l2', max_iter=1000).fit(
        X_train, y_train)
    
    # model evaluation
    train_score = accuracy_score(y_train, logistic_model.predict(X_train))
    test_score = accuracy_score(y_test, logistic_model.predict(X_test))
    duration = timer() - start
    print("Classification rate on training data: {}".format(train_score))
    print("Classification rate on test data: {}".format(test_score))
    print("Execution time: {}".format(duration))
    
    return {
        'imputation strategy': impute_strategy,
        'standardized': cols_to_standardize!=None,
        'model': logistic_model,
        'train score': train_score,
        'test score': test_score,
        'execution time (s)': duration
    }
  
# list to store models' performance  
lr_results = []

# prepare data
pima_df_cleaned = clean_data(pima_df)
cols_to_standardize=['??','BMI','??','??? ??','???','?? ??','?? ??','??? ??']

# fit logistic regression for each imputation strategy
# with and without standardizing features
for impute_strategy in ['drop_rows', 'mean', 'model_basic', 'model_progressive']:
    for cols in [None, cols_to_standardize]:
        result = logistic_regression(pima_df_cleaned, impute_strategy=impute_strategy, cols_to_standardize=cols)
        lr_results.append(result)

# display logistic regression performance
lr_results_df = pd.DataFrame(lr_results)
lr_results_df.drop(['model'], axis=1).drop_duplicates()

### Effect of dropping strategy on accuracy
: ???? ?? ??? ?? ???? response variable? ??? ??? ?? ? ?? ??? ??? ????.

? ?? training data? overfitting?? test data? ?? ??? ??, model_basic, model_progressive? test_score? ?? ??.

### Effect of mean / model imputation on accuracy
: ??? ??? ??? overfitting? ??? test data? ?? ??? ?????.

?? ? ???? ?? ? ? ?? ??? ??? ? ??? ?? ????. ?? k-nearest neighbors? ?? ? ???? ??? ??? ? ???? ?? ??? ? ? ???? ?????.

### Effect of imputation on training time
?? ???? ??, logistic regression ?? ??, ??? ? ???? ????.

model-based ??? ?? ?? ??? ?? ?? ????? ??? ?. ?? ??? ???? ?? ????? ??? ???? ?? ???? ??? ???? ??? ? ?? ?? ??? ??? ??. 

### Effect of imputation on inference

In [ ]:
# get index of strategies
lr_results_df = pd.DataFrame(lr_results)
strategies = lr_results_df['imputation strategy']

# get a boolean array where True => standardized
standardized = lr_results_df['standardized']
st = lambda s: ' standardized' if s else ''
coefs_ = {}
for key, value in enumerate(strategies):
    if value == 'drop_cols':
        # skip
        pass
    else:
        strategy = value + st(standardized[key])
        coefs_[strategy] = lr_results_df['model'][key].coef_[0]
coef_df = pd.DataFrame(data=coefs_, index=predictors)
coef_df.T

? ???? ? ?????? ?? ??? ???? ??? ??? ???? ????.

?? ??? ?? ??? ??? ??? ????.

0? ??? ??? ?? ??? ? ??? ??.

? ???? ?? ?? ?? ? ??? ??? ????.

- ????? ???? ???? ? ??.
- ?/?? ????? ?, ????, ??? ????? ? ?? ? ?? ????.
- ???? ?? ???? ?? ??? ??? ?? ??? ???. ???? ?? ????? ???? ?? ?? ??? missing values? ?? ???. ???? ???? MCAR? ???? ?? ????.

??? ?? ?? ???? ??? ??? ? ?? ???? ??? ?? ??, ?? ? ?? ?? ??? ??? ?????. 

In [ ]:
coef_perc_df = coef_df.copy()
cols = coef_df.columns.difference(['drop_rows']).values
for col in cols:
    coef_perc_df[col] = np.round(100*(coef_df[col]/coef_df['drop_rows']-1))
coef_perc_df[['drop_rows','mean','model_basic','model_progressive']]

? ?? ?? ?? ? ???? ?? ?? ? ???? ?? ?? ? ???? ??? ?? ?? ???? ?????.

?? ?? ?? ??? ???? ? ??? ?? ? ? ?? ?? ??? ??? ?????.

??, ?? ? ??? ? ??? ?? / ?? ?? ?? ???? ?? ?? ??? ??? ????.

?? ??? ????? ?? ??? ????.

## Conclusion

- ???? ???? ?? ? ?? ???? ?? Machine Learning ????? ?? ?? ??

- MCAR, MAR, MNAR ??? ?? ???? ???? ?? ?? ? ??? ??? ??? ?? ? ?? ??? ???

- ?? ? ???? ???? ??? ??? ??? ?? ??? ?? ??? ?? ??????.